In [ ]:
%pip install -q huggingface_hub tqdm pandas matplotlib

In [ ]:
import os, re, json, math, random, hashlib, zipfile, gc, time, pickle, warnings, sys, copy
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

SMOKE_TEST = False
SEED = 42

RUN_LIST = ["E1_full_ft", "E2_lora", "E3_lorma_plus", "E4_distill"]
RESUME_COMPLETED_RUNS = True
WRITE_SUBMISSIONS = True

PHASE1_CKPT_OVERRIDE = None
PHASE1_ARCH_KEYWORD = "encoder_decoder_transformer_rope_gqa_rmsnorm"
MIN_PARAMS, MAX_PARAMS = 100e6, 300e6
TOKENIZER_PATH_OVERRIDE = None
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATA_SOURCE = "hf"
LOCAL_DATA_DIR = None

OUTPUT_DIR = Path("/kaggle/working/phase2_encdec_lner") if Path("/kaggle/working").exists() \
    else Path("phase2_encdec_lner")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "runs").mkdir(exist_ok=True)

DEV_FRACTION = 0.15
DEV_STRATEGY = "random"
WINDOW = 1024
XL_MEMORY = 0
EVAL_STRIDE = 512
TRAIN_NEG_KEEP = 0.35
PRETOK_MODE = "auto"
PREPEND_EOS = False
MAX_DEV_DOCS = None

HEAD = "bilstm_crf"
LSTM_HIDDEN = 256
LSTM_LAYERS = 2
USE_PIECE_FEATURES = True
FEAT_EMB_DIM = 8
CRF_AUX_CE_WEIGHT = 0.5
RARE_OVERSAMPLE_CAP = 4.0

RUN_TAPT = True
TAPT_EPOCHS = 3
TAPT_LR = 2e-4
TAPT_BLOCK = 768
TAPT_SOURCE = 512
TAPT_BATCH = 4
TAPT_INCLUDE_TEST_TEXT = False
TAPT_MAX_TOKENS_PER_EPOCH = None

BATCH_SIZE = 4
GRAD_ACCUM = 4
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
HEAD_DROPOUT = 0.1
BACKBONE_DROPOUT = 0.1
USE_GRAD_CHECKPOINTING = False
NUM_WORKERS = 0
ADAPTER_TARGETS = ("self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.out_proj",
                   "cross_attn.q_proj", "cross_attn.k_proj", "cross_attn.v_proj", "cross_attn.out_proj",
                   "ff.gate_proj", "ff.up_proj", "ff.down_proj", "ff.net.0", "ff.net.2")

EXPERIMENTS = {
    "E1_full_ft": dict(
        method="full", lr=1e-4, head_lr=1e-3, epochs=15, patience=4,
        description="Full fine-tuning of the (task-adapted) encoder-decoder + BiLSTM-CRF head."),
    "E2_lora": dict(
        method="lora", rank=16, alpha=32, lora_dropout=0.05, lr=1e-3, head_lr=1e-3, epochs=25, patience=5,
        description="LoRA (additive low-rank update) on all attention/FFN projections; backbone frozen."),
    "E3_lorma_plus": dict(
        method="lorma_plus", rank=16, alpha=16, lr=5e-4, head_lr=1e-3, epochs=25, patience=5,
        description="LoRMA+ (low-rank MULTIPLICATIVE update with additive rank inflation): W=(I+a/r*BA)W0."),
    "E4_distill": dict(
        method="distill", student_encoder_layers=7, student_decoder_layers=3, teacher_ensemble=1,
        kd_temperature=2.0, kd_alpha=0.5, kd_hidden_weight=0.2,
        lr=1e-4, head_lr=1e-3, epochs=15, patience=4,
        description="Knowledge distillation: best fine-tuned 7+7 enc-dec teacher (~147M) -> 7+3 student (~109M, above the 100M floor)."),
}

EXPERIMENT_DEFAULTS = dict(
    head=None, bidirectional=False, class_weight="none", label_smoothing=0.0, window=None,
    batch_size=None, grad_accum=None, dropout=None, use_amp=True,
)

if SMOKE_TEST:
    WINDOW, EVAL_STRIDE, BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 96, 48, 4, 1, 8
    XL_MEMORY = 0
    TAPT_BLOCK, TAPT_SOURCE = 96, 64
    MIN_PARAMS = 0
    DEV_FRACTION = 0.3
    TAPT_EPOCHS = 1
    LSTM_HIDDEN = 32
    for _cfg in EXPERIMENTS.values():
        _cfg["epochs"] = 2
        _cfg["patience"] = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          "| memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
elif not SMOKE_TEST:
    raise RuntimeError("Enable a GPU accelerator (Kaggle: Settings > Accelerator > GPU T4/P100).")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def save_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

def make_grad_scaler(enabled):
    if not enabled:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)

In [ ]:
class ByteBPE:
    EOW = 260

    def __init__(self):
        self.specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos, self.stoi, self.payload, self.merges = [], {}, {}, []
        self.vocab_size = 0
        self._ranks = {}
        self._cache = {}

    @classmethod
    def from_json(cls, payload):
        t = cls()
        t.itos = t.specials + [f"<0x{i:02X}>" for i in range(256)] + ["</w>"]
        t.stoi = {s: i for i, s in enumerate(t.itos)}
        t.payload = {i: bytes([i - 4]) for i in range(4, 260)}
        t.payload[260] = b""
        t.merges = [tuple(map(int, p)) for p in payload["merges"]]
        for pair in t.merges:
            new_id = len(t.itos)
            symbol = f"<{pair[0]}+{pair[1]}>"
            t.itos.append(symbol)
            t.stoi[symbol] = new_id
            t.payload[new_id] = t.payload.get(pair[0], b"") + t.payload.get(pair[1], b"")
        t.vocab_size = len(t.itos)
        if t.itos != payload.get("itos"):
            raise ValueError("Saved ByteBPE vocabulary and merge list are inconsistent.")
        t._ranks = {pair: r for r, pair in enumerate(t.merges)}
        return t

    def encode_word_reference(self, word):
        seq = [b + 4 for b in str(word).encode("utf-8")] + [self.EOW]
        for rank, pair in enumerate(self.merges):
            out, j = [], 0
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == pair:
                    out.append(261 + rank); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        return seq

    def encode_piece(self, piece, add_eow=True):
        key = (piece, add_eow)
        hit = self._cache.get(key)
        if hit is not None:
            return hit
        seq = [b + 4 for b in piece.encode("utf-8")]
        if add_eow:
            seq.append(self.EOW)
        ranks = self._ranks
        while len(seq) > 1:
            best_rank, best_pair = None, None
            for i in range(len(seq) - 1):
                r = ranks.get((seq[i], seq[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, (seq[i], seq[i + 1])
            if best_pair is None:
                break
            out, j, new_id = [], 0, 261 + best_rank
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == best_pair:
                    out.append(new_id); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        if len(self._cache) < 400_000:
            self._cache[key] = seq
        return seq

def _sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _search(patterns, roots):
    found, seen = [], set()
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            for p in sorted(root.rglob(pat)):
                rp = str(p.resolve())
                if p.is_file() and rp not in seen:
                    seen.add(rp); found.append(p)
    return found

if PHASE1_CKPT_OVERRIDE:
    ckpt_candidates = [Path(PHASE1_CKPT_OVERRIDE)]
else:
    ckpt_candidates = _search(["best_t4*checkpoint*.pt", "best*checkpoint*.pt"], SEARCH_ROOTS) + \
                      _search(["latest*checkpoint*.pt", "*.pt", "*.pth"], SEARCH_ROOTS)
_payload, PHASE1_CKPT_PATH = None, None
for cand in ckpt_candidates:
    try:
        pl = torch.load(cand, map_location="cpu", weights_only=False)
    except Exception as exc:
        print(f"  skip {cand}: {exc}"); continue
    arch = str(pl.get("architecture", "")) if isinstance(pl, dict) else ""
    cfg_ = pl.get("model_config", {}) if isinstance(pl, dict) else {}
    if (isinstance(pl, dict) and "model_state_dict" in pl and PHASE1_ARCH_KEYWORD in arch.lower()
            and "encoder_layers" in cfg_ and "decoder_layers" in cfg_):
        _payload, PHASE1_CKPT_PATH = pl, cand
        break
    print(f"  skip {cand}: architecture={arch!r} (not {PHASE1_ARCH_KEYWORD})")
    del pl
if _payload is None:
    raise FileNotFoundError(
        "No RoPE-GQA-RMSNorm encoder-decoder Phase-1 checkpoint found. Attach the Phase-1 output containing "
        "best_t4_reused_bpe32k_checkpoint.pt "
        "('Add Input') or set PHASE1_CKPT_OVERRIDE.")
print("Phase-1 checkpoint:", PHASE1_CKPT_PATH)

P1_STATE = _payload["model_state_dict"]
P1_CFG = dict(_payload["model_config"])
P1_VOCAB = int(_payload.get("vocab_size", P1_CFG["vocab_size"]))
P1_TOKENIZER_SHA = _payload.get("tokenizer_sha256")
P1_ARCH = _payload.get("architecture")
P1_CFG.setdefault("rope_base", 10_000.0)
P1_CFG.setdefault("rms_norm_eps", 1e-6)
P1_CFG.setdefault("feed_forward", "swiglu")
print("Phase-1 architecture:", P1_ARCH)
print("Phase-1 config:", {k: P1_CFG.get(k) for k in ("d_model", "query_heads", "kv_heads", "encoder_layers", "decoder_layers",
                                                     "ff_hidden", "feed_forward", "encoder_max_length", "vocab_size",
                                                     "source_copy_head")})
assert P1_CFG.get("normalization", "RMSNorm") == "RMSNorm", "this notebook re-declares the RMSNorm variant"
del _payload
gc.collect()

if TOKENIZER_PATH_OVERRIDE:
    tok_candidates = [Path(TOKENIZER_PATH_OVERRIDE)]
else:
    tok_candidates = _search(["legal_bpe_tokenizer*.json", "*bpe*.json", "*tokenizer*.json"], SEARCH_ROOTS)

TOKENIZER, TOKENIZER_PATH, TOKENIZER_SHA256 = None, None, None
tok_table = []
for cand in tok_candidates:
    try:
        raw = json.loads(cand.read_text(encoding="utf-8"))
        if not (isinstance(raw, dict) and "merges" in raw and "itos" in raw):
            continue
        sha = _sha256_file(cand)
        tok_table.append((cand, int(raw.get("vocab_size", len(raw["itos"]))), sha, raw))
    except Exception:
        continue

print("Tokenizer files found:")
for cand, vs, sha, _ in tok_table:
    print(f"  {cand}  vocab={vs}  sha256={sha[:12]}")

chosen = [t for t in tok_table if P1_TOKENIZER_SHA and t[2] == P1_TOKENIZER_SHA]
how = "sha256 match with the Phase-1 checkpoint"
if not chosen:
    chosen = [t for t in tok_table if t[1] == P1_VOCAB]
    how = f"vocab size match ({P1_VOCAB}); sha256 differs/unknown - verify this is your Phase-1 tokenizer file"
if not chosen:
    raise FileNotFoundError(
        f"No tokenizer with vocab size {P1_VOCAB} found. Attach your legal_bpe_tokenizer JSON for that size "
        "(or set TOKENIZER_PATH_OVERRIDE). This notebook never trains a tokenizer.")
TOKENIZER_PATH, _, TOKENIZER_SHA256, _raw = chosen[0]
TOKENIZER = ByteBPE.from_json(_raw)
del tok_table, _raw
print(f"Using tokenizer: {TOKENIZER_PATH} ({how})")

PAD_ID, UNK_ID, BOS_ID, EOS_ID = (TOKENIZER.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
EOW_ID = ByteBPE.EOW
VOCAB_SIZE = TOKENIZER.vocab_size
assert VOCAB_SIZE == P1_VOCAB, f"Tokenizer vocab {VOCAB_SIZE} != checkpoint vocab {P1_VOCAB}"
print("Vocab size:", VOCAB_SIZE, "| ids:", dict(pad=PAD_ID, unk=UNK_ID, bos=BOS_ID, eos=EOS_ID, eow=EOW_ID))

_rng = random.Random(0)
_words = ["The", "Supreme", "Court", "of", "India,", "Section", "302", "IPC.", "(1995)", "Crl.A.No.123/2005",
          "appellant-State", "Hon'ble", "₹5,000", "naïve"]
for _w in _words:
    assert TOKENIZER.encode_piece(_w, True) == TOKENIZER.encode_word_reference(_w), _w
print("Fast BPE encoder == Phase-1 reference encoder on self-check words.")

In [ ]:
FILE_NAMES = ["fold1-text.json", "fold1-gold.json", "fold2-text.json", "fold2-gold.json",
              "labels.txt", "fold3-text.json"]
DEFAULT_LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC",
                  "DATE", "CASENO"]

def locate_phase2_files():
    paths = {}
    if DATA_SOURCE == "local":
        roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else SEARCH_ROOTS
        for name in FILE_NAMES:
            hits = _search([name], roots)
            if not hits:
                raise FileNotFoundError(f"{name} not found under {roots}")
            paths[name] = hits[0]
        return paths

    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face read token: ")
    from huggingface_hub import HfApi, hf_hub_download
    repo_files = HfApi().list_repo_files(DATASET_ID, repo_type="dataset", token=token)
    for name in FILE_NAMES:
        hits = [f for f in repo_files if f.endswith("/" + name) or f == name]
        pref = [f for f in hits if "phase_2" in f.lower() or "lner" in f.lower()]
        hits = pref or hits
        if not hits:
            raise FileNotFoundError(f"{name} not found in {DATASET_ID}. Files: {repo_files[:30]}")
        paths[name] = Path(hf_hub_download(DATASET_ID, hits[0], repo_type="dataset", token=token))
    return paths

P2_PATHS = locate_phase2_files()
for k, v in P2_PATHS.items():
    print(f"{k:16s} -> {v}")

def load_text_json(path):
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, str):
                out[str(k)] = v
            elif isinstance(v, dict) and isinstance(v.get("text"), str):
                out[str(k)] = v["text"]
            else:
                raise ValueError(f"Unexpected text-json value for id {k}: {type(v)}")
    elif isinstance(obj, list):
        for i, rec in enumerate(obj):
            rid = rec.get("id", rec.get("doc_id", rec.get("document_id", i)))
            out[str(rid)] = rec["text"]
    else:
        raise ValueError("Unexpected text-json layout")
    return out

_raw_labels = [ln.strip() for ln in Path(P2_PATHS["labels.txt"]).read_text(encoding="utf-8").splitlines() if ln.strip()]
LABELS = []
for _l in _raw_labels:
    _l = re.sub(r"^[BI]-", "", _l)
    if _l not in LABELS and _l.upper() not in ("O", "OUTSIDE"):
        LABELS.append(_l)
if not LABELS:
    LABELS = DEFAULT_LABELS
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
NUM_TAGS = 1 + 2 * NUM_LABELS
TAG_NAMES = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
print(f"{NUM_LABELS} entity labels -> {NUM_TAGS} BIO tags:", LABELS)

def parse_annotation(a):
    start = end = label = text = None
    if isinstance(a, dict):
        def pick(keys):
            for k in keys:
                if k in a and a[k] is not None:
                    return a[k]
            return None
        start = pick(("start", "start_char", "start_offset", "begin", "s"))
        end = pick(("end", "end_char", "end_offset", "e"))
        label = pick(("label", "type", "entity", "tag", "category"))
        text = pick(("text", "entity_text", "span", "mention"))
    elif isinstance(a, (list, tuple)):
        ints = [x for x in a if isinstance(x, int) and not isinstance(x, bool)]
        strs = [x for x in a if isinstance(x, str)]
        if len(ints) >= 2:
            start, end = ints[0], ints[1]
        for s in strs:
            if re.sub(r"^[BI]-", "", s) in LABEL2ID:
                label = s; break
        rest = [s for s in strs if s != label]
        text = rest[0] if rest else None
    if start is None or end is None or label is None:
        raise ValueError(f"Cannot parse annotation: {a!r}")
    label = re.sub(r"^[BI]-", "", str(label))
    return int(start), int(end), label, text

def load_gold_json(path):
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    gold = {}
    for doc_id, anns in obj.items():
        if isinstance(anns, dict):
            for key in ("entities", "annotations", "labels", "spans"):
                if key in anns:
                    anns = anns[key]; break
        gold[str(doc_id)] = [parse_annotation(a) for a in anns]
    return gold

texts, gold_raw = {}, {}
for fold in ("fold1", "fold2"):
    t = load_text_json(P2_PATHS[f"{fold}-text.json"])
    g = load_gold_json(P2_PATHS[f"{fold}-gold.json"])
    missing = set(t) - set(g)
    if missing:
        print(f"[{fold}] {len(missing)} docs without gold entry (treated as no entities)")
    for d in t:
        texts[d] = t[d]; gold_raw[d] = g.get(d, []);
    print(f"[{fold}] docs={len(t):,} entities={sum(len(v) for v in g.values()):,}")
FOLD_OF = {}
for fold in ("fold1", "fold2"):
    for d in load_text_json(P2_PATHS[f"{fold}-text.json"]):
        FOLD_OF[d] = fold
test_texts = load_text_json(P2_PATHS["fold3-text.json"])
print(f"[test fold3] docs={len(test_texts):,} (no annotations)")

match_excl = match_incl = with_text = 0
for d, anns in gold_raw.items():
    tx = texts[d]
    for s, e, l, t in anns:
        if t is None:
            continue
        with_text += 1
        match_excl += tx[s:e] == t
        match_incl += tx[s:e + 1] == t
if with_text:
    END_INCLUSIVE = match_incl > match_excl
    print(f"Offset check on {with_text:,} annotations: text[s:e]==entity -> {match_excl/with_text:.1%}; "
          f"text[s:e+1]==entity -> {match_incl/with_text:.1%}  => end is "
          f"{'INCLUSIVE' if END_INCLUSIVE else 'EXCLUSIVE'}")
else:
    END_INCLUSIVE = False
    print("Gold has no entity text; assuming end offset is EXCLUSIVE (python slicing). Verify against Codabench!")

_unknown = Counter(l for anns in gold_raw.values() for _, _, l, _ in anns if l not in LABEL2ID)
if _unknown:
    print("WARNING: gold labels not in labels.txt are dropped:", dict(_unknown))
gold_all = {d: [(s, e + 1 if END_INCLUSIVE else e, l) for s, e, l, _ in anns if l in LABEL2ID]
            for d, anns in gold_raw.items()}

label_counts = Counter(l for anns in gold_all.values() for _, _, l in anns)
print("\nEntity counts per label (train folds):")
for l in LABELS:
    print(f"  {l:10s} {label_counts.get(l, 0):7,d}")
lens = np.array([len(t) for t in texts.values()])
print(f"\nDocument length (chars): min={lens.min():,} median={int(np.median(lens)):,} "
      f"mean={int(lens.mean()):,} max={lens.max():,}")

all_ids = sorted(texts)
if DEV_STRATEGY == "fold2":
    dev_ids = [d for d in all_ids if FOLD_OF[d] == "fold2"]
else:
    rng = random.Random(SEED)
    shuffled = all_ids[:]
    rng.shuffle(shuffled)
    n_dev = max(1, int(round(DEV_FRACTION * len(shuffled))))
    dev_ids = sorted(shuffled[:n_dev])
dev_set = set(dev_ids)
train_ids = [d for d in all_ids if d not in dev_set]
if MAX_DEV_DOCS:
    dev_ids = dev_ids[:MAX_DEV_DOCS]
print(f"\nTrain docs: {len(train_ids):,} | Dev docs: {len(dev_ids):,} | Test docs: {len(test_texts):,}")

In [ ]:
PIECE_REGEX = {"punct": re.compile(r"\w+|[^\w\s]"), "whitespace": re.compile(r"\S+")}

def piece_spans_of(text, mode):
    return np.array([(m.start(), m.end()) for m in PIECE_REGEX[mode].finditer(text)], dtype=np.int64).reshape(-1, 2)

PUNCT_ID = {".": 1, ",": 2, "(": 3, ")": 4, "/": 5, "-": 6, ":": 7, ";": 8, "'": 9, "\u2019": 9, '"': 10,
            "\u201c": 10, "\u201d": 10, "&": 11, "[": 3, "]": 4}
FEAT_SIZES = [8, 13, 13, 10, 4, 18]

def _shape(p):
    if p.isdigit():
        return 4
    if p.isalpha():
        if p.isupper() and len(p) > 1:
            return 2
        if p[0].isupper():
            return 1
        return 0 if p.islower() else 3
    if any(c.isalnum() for c in p):
        return 5
    return 6 if len(p) == 1 else 7

def encode_document(text, mode):
    spans = piece_spans_of(text, mode)
    n = len(text)
    ids = [EOS_ID] if PREPEND_EOS else []
    readout = np.empty(len(spans), dtype=np.int64)
    feats = np.zeros((len(spans), len(FEAT_SIZES)), dtype=np.int16)
    prev_end = 0
    for i, (s, e) in enumerate(spans):
        final = (e == n) or text[e].isspace()
        piece = text[s:e]
        ids.extend(TOKENIZER.encode_piece(piece, final))
        readout[i] = len(ids) - 1
        shp = _shape(piece)
        line_start = 1 if (i == 0 or "\n" in text[prev_end:s]) else 0
        feats[i] = (shp, PUNCT_ID.get(piece, 12 if shp == 6 else 0), min(e - s, 12),
                    min(9, int(10 * s / max(1, n))), line_start * 2 + int(final), min(17, int(math.log2(s + 1))))
        prev_end = e
    return {"ids": np.asarray(ids, dtype=np.int32), "readout": readout, "pieces": spans, "feats": feats}

def make_piece_tags(text, spans_excl, pieces):
    P = len(pieces)
    tags = np.zeros(P, dtype=np.int64)
    if P == 0 or not spans_excl:
        return tags
    n = len(text)
    sid = np.full(n, -1, dtype=np.int32)
    order = sorted(range(len(spans_excl)), key=lambda i: -(spans_excl[i][1] - spans_excl[i][0]))
    for i in order:
        s, e, _ = spans_excl[i]
        s, e = max(0, s), min(n, e)
        if e > s:
            sid[s:e] = i
    inside = np.concatenate([[0], np.cumsum(sid >= 0)])
    ps, pe = pieces[:, 0], pieces[:, 1]
    covered = (inside[pe] - inside[ps]) * 2 > (pe - ps)
    mid = sid[(ps + pe - 1) // 2]
    piece_sid = np.where(covered & (mid >= 0), mid, -1)
    prev = np.concatenate([[-1], piece_sid[:-1]])
    label_of_sid = np.array([LABEL2ID[spans_excl[i][2]] for i in range(len(spans_excl))] + [0])
    li = label_of_sid[piece_sid]
    is_b = piece_sid != prev
    ent = piece_sid >= 0
    tags[ent] = np.where(is_b[ent], 1 + 2 * li[ent], 2 + 2 * li[ent])
    return tags

def tags_to_spans(tags, pieces):
    spans, cur = [], None
    for i, t in enumerate(tags):
        t = int(t)
        if t == 0:
            if cur: spans.append(cur); cur = None
            continue
        k, is_b = (t - 1) // 2, (t - 1) % 2 == 0
        if is_b or cur is None or cur[2] != k:
            if cur: spans.append(cur)
            cur = [i, i, k]
        else:
            cur[1] = i
    if cur:
        spans.append(cur)
    return [(int(pieces[a, 0]), int(pieces[b, 1]), LABELS[k]) for a, b, k in spans]

def oracle_audit(mode, doc_ids, max_docs=150):
    tot = rec = pieces_total = 0
    for d in doc_ids[:max_docs]:
        text = texts[d]
        pieces = piece_spans_of(text, mode)
        tags = make_piece_tags(text, gold_all[d], pieces)
        got = set(tags_to_spans(tags, pieces))
        want = set(gold_all[d])
        tot += len(want); rec += len(want & got); pieces_total += len(pieces)
    return rec / max(1, tot), pieces_total / max(1, min(max_docs, len(doc_ids)))

print("Oracle exact-boundary recall (upper bound for strict F1 with each pre-tokeniser):")
audit = {}
for mode in ("punct", "whitespace"):
    r, ppd = oracle_audit(mode, train_ids)
    audit[mode] = r
    print(f"  {mode:11s} recall={r:.4f}  pieces/doc={ppd:,.0f}")
if PRETOK_MODE == "auto":
    PRETOK = "punct" if audit["punct"] >= audit["whitespace"] - 1e-9 else "whitespace"
else:
    PRETOK = PRETOK_MODE
print("Selected pre-tokeniser:", PRETOK)

ENC_CACHE = OUTPUT_DIR / f"encoded_v2_{PRETOK}_{TOKENIZER_SHA256[:8]}_{len(texts)}_{len(test_texts)}_{int(PREPEND_EOS)}.pkl"
if ENC_CACHE.exists():
    with ENC_CACHE.open("rb") as f:
        ENC = pickle.load(f)
    print("Loaded cached encodings:", ENC_CACHE.name)
else:
    ENC = {"train": {}, "test": {}}
    for d in tqdm(all_ids, desc="Encoding train/dev docs"):
        e = encode_document(texts[d], PRETOK)
        e["tags"] = make_piece_tags(texts[d], gold_all[d], e["pieces"])
        ENC["train"][d] = e
    for d in tqdm(sorted(test_texts), desc="Encoding test docs"):
        ENC["test"][d] = encode_document(test_texts[d], PRETOK)
    with ENC_CACHE.open("wb") as f:
        pickle.dump(ENC, f, protocol=4)

tok_counts = np.array([len(e["ids"]) for e in ENC["train"].values()])
print(f"Tokens/doc: median={int(np.median(tok_counts)):,} max={tok_counts.max():,} | total train tokens={tok_counts.sum():,}")
print(f"Phase-1 encoder context: {P1_CFG['encoder_max_length']} | fine-tuning window: {WINDOW} (+<s>, </s>)")
assert WINDOW + 2 <= P1_CFG["encoder_max_length"]

TAG_FREQ = np.zeros(NUM_TAGS, dtype=np.int64)
for d in train_ids:
    TAG_FREQ += np.bincount(ENC["train"][d]["tags"], minlength=NUM_TAGS)
print(f"Piece tags: O={TAG_FREQ[0]:,} ({TAG_FREQ[0]/TAG_FREQ.sum():.1%}) | entity pieces={TAG_FREQ[1:].sum():,}")

TRAIN_LABEL_COUNTS = Counter(l for d in train_ids for _, _, l in gold_all[d])
_mx = max(TRAIN_LABEL_COUNTS.values()) if TRAIN_LABEL_COUNTS else 1
LABEL_OVERSAMPLE = np.array([min(RARE_OVERSAMPLE_CAP, math.sqrt(_mx / max(1, TRAIN_LABEL_COUNTS.get(l, 0))))
                             if TRAIN_LABEL_COUNTS.get(l, 0) else 1.0 for l in LABELS])
print("Window oversampling factor per label:", dict(zip(LABELS, np.round(LABEL_OVERSAMPLE, 2))))

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = float(eps)

    def forward(self, x):
        dt = x.dtype
        xf = x.float()
        out = (xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)).to(dtype=dt)
        return out * self.weight.to(dtype=dt)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10_000.0):
        super().__init__()
        inv_freq = 1.0 / (float(base) ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        fr = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", fr.cos().repeat_interleave(2, dim=-1)[None, None], persistent=False)
        self.register_buffer("sin_cached", fr.sin().repeat_interleave(2, dim=-1)[None, None], persistent=False)

    @staticmethod
    def rotate_half(x):
        return torch.stack((-x[..., 1::2], x[..., 0::2]), dim=-1).flatten(-2)

    def forward(self, x):
        n = x.size(-2)
        cos = self.cos_cached[:, :, :n].to(device=x.device, dtype=x.dtype)
        sin = self.sin_cached[:, :, :n].to(device=x.device, dtype=x.dtype)
        return x * cos + self.rotate_half(x) * sin

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout, use_rope, rope_base):
        super().__init__()
        self.n_heads, self.n_kv_heads = int(n_heads), int(n_kv_heads)
        self.groups = self.n_heads // self.n_kv_heads
        self.head_dim = d_model // self.n_heads
        self.dropout = float(dropout)
        self.use_rope = bool(use_rope)
        kv = self.n_kv_heads * self.head_dim
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, kv)
        self.v_proj = nn.Linear(d_model, kv)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len, rope_base) if self.use_rope else None

    def forward(self, x, kv_states=None, key_padding_mask=None, causal=False):
        b, t, w = x.shape
        kvs = x if kv_states is None else kv_states
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(kvs).view(b, kvs.size(1), self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(kvs).view(b, kvs.size(1), self.n_kv_heads, self.head_dim).transpose(1, 2)
        if self.use_rope:
            q, k = self.rope(q), self.rope(k)
        k = k.repeat_interleave(self.groups, dim=1)
        v = v.repeat_interleave(self.groups, dim=1)
        allowed = None if key_padding_mask is None else key_padding_mask[:, None, None, :].bool()
        if causal:
            c = torch.ones(t, k.size(-2), dtype=torch.bool, device=x.device).tril()[None, None]
            allowed = c if allowed is None else allowed & c
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(y.transpose(1, 2).contiguous().view(b, t, w))

class FeedForward(nn.Module):
    def __init__(self, d_model, hidden, dropout, swiglu):
        super().__init__()
        self.swiglu = bool(swiglu)
        if self.swiglu:
            self.gate_proj = nn.Linear(d_model, hidden, bias=False)
            self.up_proj = nn.Linear(d_model, hidden, bias=False)
            self.down_proj = nn.Linear(hidden, d_model, bias=False)
            self.drop = nn.Dropout(dropout)
        else:
            self.net = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x):
        if self.swiglu:
            return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d, h, kvh, ff, ctx, dropout, swiglu, rope_base, eps):
        super().__init__()
        self.norm_attn, self.norm_ff = RMSNorm(d, eps), RMSNorm(d, eps)
        self.self_attn = GroupedQueryAttention(d, h, kvh, ctx, dropout, True, rope_base)
        self.ff = FeedForward(d, ff, dropout, swiglu)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = x + self.resid_dropout(self.self_attn(self.norm_attn(x), key_padding_mask=mask, causal=False))
        return x + self.resid_dropout(self.ff(self.norm_ff(x)))

class DecoderLayer(nn.Module):
    def __init__(self, d, h, kvh, ff, ctx, dropout, swiglu, rope_base, eps):
        super().__init__()
        self.norm_self, self.norm_cross, self.norm_ff = RMSNorm(d, eps), RMSNorm(d, eps), RMSNorm(d, eps)
        self.self_attn = GroupedQueryAttention(d, h, kvh, ctx, dropout, True, rope_base)
        self.cross_attn = GroupedQueryAttention(d, h, kvh, ctx, dropout, False, rope_base)
        self.ff = FeedForward(d, ff, dropout, swiglu)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, memory, mem_mask, dec_mask):
        x = x + self.resid_dropout(self.self_attn(self.norm_self(x), key_padding_mask=dec_mask, causal=True))
        x = x + self.resid_dropout(self.cross_attn(self.norm_cross(x), kv_states=memory, key_padding_mask=mem_mask))
        return x + self.resid_dropout(self.ff(self.norm_ff(x)))

class EncDecBackbone(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_kv_heads, n_enc, n_dec, ff_hidden, context, dropout,
                 swiglu=True, rope_base=10_000.0, eps=1e-6):
        super().__init__()
        self.d_model = d_model
        self.out_dim = 2 * d_model
        self.shared_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        args = (d_model, n_heads, n_kv_heads, ff_hidden, context, dropout, swiglu, rope_base, eps)
        self.encoder_layers = nn.ModuleList([EncoderLayer(*args) for _ in range(n_enc)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(*args) for _ in range(n_dec)])
        self.encoder_norm = RMSNorm(d_model, eps)
        self.decoder_norm = RMSNorm(d_model, eps)

    @property
    def blocks(self):
        return list(self.encoder_layers) + list(self.decoder_layers)

    def final_norms(self):
        return [self.encoder_norm, self.decoder_norm]

    def encode(self, src, mask, return_hidden=False, grad_ckpt=False):
        x = self.shared_embedding(src)
        hs = []
        for layer in self.encoder_layers:
            x = checkpoint(layer, x, mask, use_reentrant=False) if grad_ckpt else layer(x, mask)
            if return_hidden:
                hs.append(x)
        return self.encoder_norm(x), hs

    def decode(self, dec, memory, mem_mask, dec_mask, return_hidden=False, grad_ckpt=False):
        x = self.shared_embedding(dec)
        hs = []
        for layer in self.decoder_layers:
            if grad_ckpt:
                x = checkpoint(layer, x, memory, mem_mask, dec_mask, use_reentrant=False)
            else:
                x = layer(x, memory, mem_mask, dec_mask)
            if return_hidden:
                hs.append(x)
        return self.decoder_norm(x), hs

    @staticmethod
    def wrap(input_ids, attn_mask):
        b = input_ids.size(0)
        lengths = attn_mask.sum(1)
        bos = torch.full((b, 1), BOS_ID, dtype=input_ids.dtype, device=input_ids.device)
        pad = torch.full((b, 1), PAD_ID, dtype=input_ids.dtype, device=input_ids.device)
        t_ = torch.ones((b, 1), dtype=torch.bool, device=input_ids.device)
        f_ = torch.zeros((b, 1), dtype=torch.bool, device=input_ids.device)
        enc = torch.cat([bos, input_ids.masked_fill(~attn_mask, PAD_ID), pad], 1)
        enc_mask = torch.cat([t_, attn_mask, f_], 1)
        rows = torch.arange(b, device=input_ids.device)
        enc[rows, lengths + 1] = EOS_ID
        enc_mask[rows, lengths + 1] = True
        dec = torch.cat([bos, input_ids.masked_fill(~attn_mask, PAD_ID)], 1)
        dec_mask = torch.cat([t_, attn_mask], 1)
        return enc, enc_mask, dec, dec_mask

    def forward(self, input_ids, attn_mask, return_hidden=False, grad_ckpt=False):
        enc, enc_mask, dec, dec_mask = self.wrap(input_ids, attn_mask)
        mem, enc_hs = self.encode(enc, enc_mask, return_hidden, grad_ckpt)
        dh, dec_hs = self.decode(dec, mem, enc_mask, dec_mask, return_hidden, grad_ckpt)
        T = input_ids.size(1)
        h = torch.cat([mem[:, 1:T + 1], dh[:, 1:T + 1]], dim=-1)
        if return_hidden:
            return h, [x[:, 1:T + 1] for x in enc_hs + dec_hs]
        return h

class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(num_tags, num_tags))
        self.start = nn.Parameter(torch.zeros(num_tags))
        self.end = nn.Parameter(torch.zeros(num_tags))
        allowed = np.zeros((num_tags, num_tags), dtype=bool)
        for p_ in range(num_tags):
            for n_ in range(num_tags):
                if n_ == 0 or (n_ - 1) % 2 == 0:
                    allowed[p_, n_] = True
                else:
                    k = (n_ - 1) // 2
                    allowed[p_, n_] = p_ in (1 + 2 * k, 2 + 2 * k)
        start_ok = np.array([t == 0 or (t - 1) % 2 == 0 for t in range(num_tags)])
        self.register_buffer("trans_mask", torch.tensor(np.where(allowed, 0.0, -1e4), dtype=torch.float32))
        self.register_buffer("start_mask", torch.tensor(np.where(start_ok, 0.0, -1e4), dtype=torch.float32))

    def effective(self):
        return self.trans + self.trans_mask, self.start + self.start_mask, self.end

    def nll(self, em, tags, mask):
        T, S, E = self.effective()
        B, P, K = em.shape
        tags = tags.clamp(min=0)
        mf = mask.float()
        score = S[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        if P > 1:
            emit = em[:, 1:].gather(2, tags[:, 1:, None]).squeeze(2)
            tr = T[tags[:, :-1], tags[:, 1:]]
            score = score + ((emit + tr) * mf[:, 1:]).sum(1)
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        score = score + E[last]
        alpha = S[None] + em[:, 0]
        for t in range(1, P):
            nxt = torch.logsumexp(alpha[:, :, None] + T[None] + em[:, t, None, :], dim=1)
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        logz = torch.logsumexp(alpha + E[None], dim=1)
        return logz - score

class PieceFeatures(nn.Module):
    def __init__(self, sizes, dim):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, dim) for n in sizes])
        self.out_dim = dim * len(sizes)

    def forward(self, f):
        return torch.cat([emb(f[..., i]) for i, emb in enumerate(self.embs)], dim=-1)

class NERModel(nn.Module):
    def __init__(self, backbone, num_tags, dropout=0.1, causal=True, head="bilstm_crf"):
        super().__init__()
        self.backbone = backbone
        self.head_type = head
        self.drop = nn.Dropout(dropout)
        d = backbone.out_dim
        self.feat = PieceFeatures(FEAT_SIZES, FEAT_EMB_DIM) if USE_PIECE_FEATURES else None
        in_dim = d + (self.feat.out_dim if self.feat is not None else 0)
        if head in ("bilstm", "bilstm_crf"):
            self.lstm = nn.LSTM(in_dim, LSTM_HIDDEN, num_layers=LSTM_LAYERS, batch_first=True, bidirectional=True,
                                dropout=dropout if LSTM_LAYERS > 1 else 0.0)
            out_dim = 2 * LSTM_HIDDEN + in_dim
        else:
            self.lstm = None
            out_dim = in_dim
        self.classifier = nn.Linear(out_dim, num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
        self.crf = CRF(num_tags) if head == "bilstm_crf" else None
        self.causal = False
        self.grad_ckpt = False

    def head_modules(self):
        return [m for m in (self.feat, self.lstm, self.classifier, self.crf) if m is not None]

    def forward(self, input_ids, attn_mask, piece_pos, piece_mask, piece_feats, return_hidden=False,
                mem_ids=None, mem_mask=None):
        out = self.backbone(input_ids, attn_mask, return_hidden=return_hidden,
                            grad_ckpt=self.grad_ckpt and self.training)
        h, levels = out if return_hidden else (out, None)
        idx = piece_pos.clamp(min=0)
        x = self.drop(h.gather(1, idx[..., None].expand(-1, -1, h.size(-1))))
        if self.feat is not None:
            x = torch.cat([x, self.feat(piece_feats.long()).to(x.dtype)], dim=-1)
        if self.lstm is not None:
            lengths = piece_mask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            y, _ = self.lstm(packed)
            y, _ = nn.utils.rnn.pad_packed_sequence(y, batch_first=True, total_length=piece_pos.size(1))
            x = torch.cat([self.drop(y), x.to(y.dtype)], dim=-1)
        em = self.classifier(x)
        return (em, levels) if return_hidden else em

def build_backbone(n_enc=None, n_dec=None, enc_map=None, dec_map=None, dropout=None):
    c = P1_CFG
    n_enc = n_enc or c["encoder_layers"]
    n_dec = n_dec or c["decoder_layers"]
    enc_map = enc_map or list(range(n_enc))
    dec_map = dec_map or list(range(n_dec))
    bb = EncDecBackbone(VOCAB_SIZE, c["d_model"], c["query_heads"], c["kv_heads"], n_enc, n_dec, c["ff_hidden"],
                        c["encoder_max_length"], BACKBONE_DROPOUT if dropout is None else dropout,
                        swiglu=(c.get("feed_forward", "swiglu") == "swiglu"), rope_base=c.get("rope_base", 10_000.0),
                        eps=c.get("rms_norm_eps", 1e-6))
    sd = {k: v for k, v in P1_STATE.items() if not k.startswith(("lm_head", "copy_head"))}
    new_sd = {k: v for k, v in sd.items() if not k.startswith(("encoder_layers.", "decoder_layers."))}
    for prefix, mapping in (("encoder_layers", enc_map), ("decoder_layers", dec_map)):
        for new_i, old_i in enumerate(mapping):
            pre = f"{prefix}.{old_i}."
            for k, v in sd.items():
                if k.startswith(pre):
                    new_sd[f"{prefix}.{new_i}." + k[len(pre):]] = v
    missing, unexpected = bb.load_state_dict(new_sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"Phase-1 checkpoint does not match backbone. missing={missing[:5]} unexpected={unexpected[:5]}")
    return bb

def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))

_probe = build_backbone()
P1_PARAMS = count_params(_probe)
print(f"Phase-1 encoder-decoder parameters used for tagging (tied embedding once, LM/copy heads excluded): {P1_PARAMS/1e6:.2f}M")
assert MIN_PARAMS <= P1_PARAMS <= MAX_PARAMS or SMOKE_TEST, f"Course limit: {MIN_PARAMS/1e6:.0f}M-{MAX_PARAMS/1e6:.0f}M parameters"
del _probe

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        out = base.out_features
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, out))
        self.B = nn.Parameter(torch.zeros(out, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias

class LoRMAPiLinear(nn.Module):
    def __init__(self, base, r, alpha=None, dropout=0.0):
        super().__init__()
        self.base = base
        out = base.out_features
        B = torch.randn(out, r) / math.sqrt(r)
        B[:, 0] = 1.0
        A = torch.zeros(r, out)
        A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        idx = (torch.arange(out)[None, :] - torch.arange(out)[:, None]) % out
        self.register_buffer("rot_idx", idx, persistent=False)

    def forward(self, x):
        m = torch.matmul(self.B, self.A)
        m_pi = torch.gather(m, 1, self.rot_idx)
        y = F.linear(F.linear(x, self.base.weight), m_pi)
        return y if self.base.bias is None else y + self.base.bias

ADAPTER_CLASSES = {"lora": LoRALinear, "lorma_plus": LoRMAPlusLinear, "lorma_pi": LoRMAPiLinear}

def inject_adapters(backbone, method, rank, alpha, dropout, targets=ADAPTER_TARGETS):
    cls = ADAPTER_CLASSES[method]
    n = 0
    for block in backbone.blocks:
        for path in targets:
            parts = path.split(".")
            parent = block
            try:
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                leaf = getattr(parent, parts[-1])
            except AttributeError:
                continue
            if isinstance(leaf, nn.Linear):
                setattr(parent, parts[-1], cls(leaf, rank, alpha, dropout))
                n += 1
    print(f"{method}: adapters injected into {n} linear layers")
    return n

def configure_trainable(model, cfg):
    method = cfg["method"]
    for p in model.parameters():
        p.requires_grad = False
    if method in ("full", "distill"):
        for p in model.parameters():
            p.requires_grad = True
    elif method in ADAPTER_CLASSES:
        for mod in model.backbone.modules():
            if isinstance(mod, tuple(ADAPTER_CLASSES.values())):
                mod.A.requires_grad = True
                mod.B.requires_grad = True
    elif method == "partial":
        k = cfg["train_top_layers"]
        for blk in model.backbone.blocks[-k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for norm in model.backbone.final_norms():
            for p in norm.parameters():
                p.requires_grad = True
    else:
        raise ValueError(method)
    for m in model.head_modules():
        for p in m.parameters():
            p.requires_grad = True

def build_ner_model(cfg, backbone=None):
    method = cfg["method"]
    drop = cfg["dropout"] if cfg.get("dropout") is not None else BACKBONE_DROPOUT
    if backbone is None:
        if method == "distill":
            E, D = P1_CFG["encoder_layers"], P1_CFG["decoder_layers"]
            ne = cfg.get("student_encoder_layers") or E
            nd = cfg.get("student_decoder_layers") or D
            em = [int(round(x)) for x in np.linspace(0, E - 1, ne)]
            dm = [int(round(x)) for x in np.linspace(0, D - 1, nd)]
            cfg["layer_map"] = em + [E + j for j in dm]
            cfg["student_maps"] = dict(encoder=em, decoder=dm)
            backbone = build_backbone(n_enc=ne, n_dec=nd, enc_map=em, dec_map=dm, dropout=drop)
        else:
            backbone = build_backbone(dropout=drop)
    if method in ADAPTER_CLASSES:
        inject_adapters(backbone, method, cfg["rank"], cfg.get("alpha", cfg["rank"]), cfg.get("lora_dropout", 0.0))
    cfg["head"] = cfg.get("head") or HEAD
    model = NERModel(backbone, NUM_TAGS, HEAD_DROPOUT, causal=not cfg["bidirectional"], head=cfg["head"])
    model.grad_ckpt = USE_GRAD_CHECKPOINTING
    configure_trainable(model, cfg)
    return model

def _adapter_identity_check():
    torch.manual_seed(0)
    base = nn.Linear(24, 20)
    x = torch.randn(3, 24)
    ref = base(x)
    for name, cls in ADAPTER_CLASSES.items():
        m = cls(base, 4, 4, 0.0)
        assert torch.allclose(m(x), ref, atol=1e-5), f"{name} is not identity at init"

    m = LoRMAPiLinear(base, 4)
    mp = torch.gather(m.B @ m.A, 1, m.rot_idx)
    assert torch.allclose(mp, torch.eye(20))

    p = LoRMAPlusLinear(base, 4, 4)
    nn.init.normal_(p.B, std=0.1)
    y0 = x @ base.weight.T
    want = y0 + p.scale * (y0 @ p.A.T @ p.B.T) + base.bias
    assert torch.allclose(p(x), want, atol=1e-5)
    print("Adapter identity-at-init and LoRMA+ formula checks passed.")

_adapter_identity_check()

In [ ]:
def window_pieces(e, a, b):
    r = e["readout"]
    return int(np.searchsorted(r, a, side="left")), int(np.searchsorted(r, b, side="left"))

def window_item(e, a, b, with_tags):
    i0, i1 = window_pieces(e, a, b)
    item = dict(ids=e["ids"][a:b].astype(np.int64), pos=(e["readout"][i0:i1] - a).astype(np.int64),
                feats=e["feats"][i0:i1].astype(np.int64), i0=i0, i1=i1,
                mem=e["ids"][max(0, a - XL_MEMORY):a].astype(np.int64))
    if with_tags:
        t = e["tags"][i0:i1].astype(np.int64).copy()
        if len(t) and t[0] > 0 and (t[0] - 1) % 2 == 1:
            t[0] -= 1
        item["tags"] = t
    return item

class TrainWindows(Dataset):
    def __init__(self, doc_ids, window, neg_keep, rng):
        self.items = []
        for d in doc_ids:
            e = ENC["train"][d]
            T = len(e["ids"])
            off = rng.randrange(window) if T > window else 0
            bounds = sorted(set([0, T] + ([off] if 0 < off < T else []) + list(range(off + window, T, window))))
            for a, b in zip(bounds[:-1], bounds[1:]):
                i0, i1 = window_pieces(e, a, b)
                if i1 <= i0:
                    continue
                tags = e["tags"][i0:i1]
                ent = tags[tags > 0]
                if len(ent) == 0:
                    if rng.random() < neg_keep:
                        self.items.append((d, a, b))
                    continue
                w = float(LABEL_OVERSAMPLE[np.unique((ent - 1) // 2)].max())
                reps = int(w) + (1 if rng.random() < w - int(w) else 0)
                self.items.extend([(d, a, b)] * max(1, reps))
        rng.shuffle(self.items)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        d, a, b = self.items[i]
        return window_item(ENC["train"][d], a, b, True)

def collate(batch):
    T = max(len(x["ids"]) for x in batch)
    P = max(len(x["pos"]) for x in batch)
    B = len(batch)
    ids = torch.full((B, T), PAD_ID, dtype=torch.long)
    mask = torch.zeros((B, T), dtype=torch.bool)
    pos = torch.zeros((B, P), dtype=torch.long)
    pmask = torch.zeros((B, P), dtype=torch.bool)
    feats = torch.zeros((B, P, len(FEAT_SIZES)), dtype=torch.long)
    tags = torch.full((B, P), -100, dtype=torch.long)
    M = max(len(x["mem"]) for x in batch)
    mem_ids = torch.full((B, M), PAD_ID, dtype=torch.long)
    mem_mask = torch.zeros((B, M), dtype=torch.bool)
    for i, x in enumerate(batch):
        k = len(x["mem"])
        if k:
            mem_ids[i, M - k:] = torch.from_numpy(x["mem"]); mem_mask[i, M - k:] = True
        n, m = len(x["ids"]), len(x["pos"])
        ids[i, :n] = torch.from_numpy(x["ids"]); mask[i, :n] = True
        pos[i, :m] = torch.from_numpy(x["pos"]); pmask[i, :m] = True
        feats[i, :m] = torch.from_numpy(x["feats"])
        if "tags" in x:
            tags[i, :m] = torch.from_numpy(x["tags"])
    return dict(ids=ids, mask=mask, pos=pos, pmask=pmask, feats=feats, tags=tags, mem_ids=mem_ids, mem_mask=mem_mask)

def to_device(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

def eval_windows(T, window, stride):
    if T <= window:
        return [(0, T)]
    starts = list(range(0, T - window + 1, stride))
    if starts[-1] + window < T:
        starts.append(T - window)
    return [(s, s + window) for s in starts]

@torch.no_grad()
def predict_doc_scores(model, e, window=None, stride=None, batch_size=None):
    window, stride, batch_size = window or WINDOW, stride or EVAL_STRIDE, batch_size or EVAL_BATCH_SIZE
    model.eval()
    P = len(e["readout"])
    acc = np.zeros((P, NUM_TAGS), dtype=np.float32)
    wsum = np.zeros(P, dtype=np.float32)
    wins = eval_windows(len(e["ids"]), window, stride)
    for k in range(0, len(wins), batch_size):
        items = [window_item(e, a, b, False) for a, b in wins[k:k + batch_size]]
        keep = [(it, ab) for it, ab in zip(items, wins[k:k + batch_size]) if it["i1"] > it["i0"]]
        if not keep:
            continue
        batch = to_device(collate([it for it, _ in keep]))
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            em = model(batch["ids"], batch["mask"], batch["pos"], batch["pmask"], batch["feats"],
                       mem_ids=batch["mem_ids"], mem_mask=batch["mem_mask"])
        em = em.float().cpu().numpy()
        for j, (it, (a, b)) in enumerate(keep):
            m = it["i1"] - it["i0"]
            tok = it["pos"]
            w = np.minimum(np.minimum(tok + 1, (b - a) - tok), 64).astype(np.float32)
            acc[it["i0"]:it["i1"]] += em[j, :m] * w[:, None]
            wsum[it["i0"]:it["i1"]] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]

def _transition_mask():
    allowed = np.zeros((NUM_TAGS, NUM_TAGS), dtype=bool)
    for prev in range(NUM_TAGS):
        for nxt in range(NUM_TAGS):
            if nxt == 0 or (nxt - 1) % 2 == 0:
                allowed[prev, nxt] = True
            else:
                k = (nxt - 1) // 2
                allowed[prev, nxt] = prev in (1 + 2 * k, 2 + 2 * k)
    start_ok = np.array([t == 0 or (t - 1) % 2 == 0 for t in range(NUM_TAGS)])
    return allowed, start_ok

TRANS_ALLOWED, START_OK = _transition_mask()
CONSTRAINT_PACK = (np.where(TRANS_ALLOWED, 0.0, -1e4).astype(np.float32),
                   np.where(START_OK, 0.0, -1e4).astype(np.float32), np.zeros(NUM_TAGS, dtype=np.float32))

def model_transitions(model):
    if getattr(model, "crf", None) is None:
        return CONSTRAINT_PACK
    T, S, E = model.crf.effective()
    return tuple(x.detach().float().cpu().numpy() for x in (T, S, E))

def viterbi_tags(sc, pack=CONSTRAINT_PACK):
    trans, start, end = pack
    P = sc.shape[0]
    if P == 0:
        return np.zeros(0, dtype=np.int64)
    score = sc[0] + start
    back = np.zeros((P, NUM_TAGS), dtype=np.int64)
    for t in range(1, P):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + sc[t]
    score = score + end
    tags = np.empty(P, dtype=np.int64)
    tags[-1] = score.argmax()
    for t in range(P - 1, 0, -1):
        tags[t - 1] = back[t, tags[t]]
    return tags

def decode_spans(sc, pieces, method="viterbi", pack=CONSTRAINT_PACK):
    tags = viterbi_tags(sc, pack) if method == "viterbi" else sc.argmax(-1)
    return tags_to_spans(tags, pieces)

def strict_scores(pred, gold, labels=LABELS):
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in gold:
        g = set(gold[d]); p = set(pred.get(d, []))
        for s in g & p: tp[s[2]] += 1
        for s in p - g: fp[s[2]] += 1
        for s in g - p: fn[s[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F1 = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = dict(precision=P, recall=R, f1=F1, support=tp[l] + fn[l], predicted=tp[l] + fp[l])
    used = [l for l in labels if per[l]["support"] > 0 or per[l]["predicted"] > 0]
    macro = lambda k: float(np.mean([per[l][k] for l in used])) if used else 0.0
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mp = TP / (TP + FP) if TP + FP else 0.0
    mr = TP / (TP + FN) if TP + FN else 0.0
    return dict(macro_precision=macro("precision"), macro_recall=macro("recall"), macro_f1=macro("f1"),
                micro_precision=mp, micro_recall=mr, micro_f1=2 * mp * mr / (mp + mr) if mp + mr else 0.0,
                per_label=per)

def evaluate_docs(model, doc_ids, methods=("viterbi",), keep_scores=False):
    pack = model_transitions(model)
    preds = {m: {} for m in methods}
    scores = {}
    n_tokens, t0 = 0, time.time()
    for d in doc_ids:
        e = ENC["train"][d]
        sc = predict_doc_scores(model, e)
        n_tokens += len(e["ids"])
        for m in methods:
            preds[m][d] = decode_spans(sc, e["pieces"], m, pack)
        if keep_scores:
            scores[d] = sc
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    dt = time.time() - t0
    gold = {d: gold_all[d] for d in doc_ids}
    out = {m: strict_scores(preds[m], gold) for m in methods}
    return out, preds, dict(tokens=n_tokens, seconds=dt, tokens_per_s=n_tokens / max(dt, 1e-9)), scores

In [ ]:
TRAINED_MODELS = {}
RESULTS = {}

def make_cfg(name):
    cfg = dict(EXPERIMENT_DEFAULTS)
    cfg.update(copy.deepcopy(EXPERIMENTS[name]))
    cfg["name"] = name
    return cfg

def class_weight_tensor(cfg):
    if cfg["class_weight"] == "none":
        return None
    f = TAG_FREQ.astype(np.float64) + 1.0
    w = np.clip(np.sqrt(f[0] / f), 1.0, 10.0)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}

def spans_to_submission(spans_by_doc, id_list):
    out = {}
    for d in id_list:
        out[d] = [[int(s), int(e - 1 if END_INCLUSIVE else e), l] for s, e, l in spans_by_doc.get(d, [])]
    return out

def validate_submission(path, expected_ids, text_by_id):
    sub = json.loads(Path(path).read_text(encoding="utf-8"))
    assert set(sub) == set(expected_ids), "prediction ids must equal the official test ids exactly"
    n = 0
    for d, spans in sub.items():
        L = len(text_by_id[d])
        for sp in spans:
            assert len(sp) in (3, 4), sp
            s, e, l = sp[0], sp[1], sp[2]
            assert isinstance(s, int) and isinstance(e, int) and l in LABEL2ID, sp
            assert 0 <= s <= e <= L, (d, sp, L)
            n += 1
    return n

def write_submission(run_dir, spans_by_doc, id_list, text_by_id):
    sub = spans_to_submission(spans_by_doc, id_list)
    pj = Path(run_dir) / "prediction.json"
    pj.write_text(json.dumps(sub, ensure_ascii=False), encoding="utf-8")
    n = validate_submission(pj, id_list, text_by_id)
    zp = Path(run_dir) / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    return zp, n

def predict_test(model, decoder):
    pack = model_transitions(model)
    out, scores = {}, {}
    for d in sorted(ENC["test"]):
        sc = predict_doc_scores(model, ENC["test"][d])
        scores[d] = sc
        out[d] = decode_spans(sc, ENC["test"][d]["pieces"], decoder, pack)
    return out, scores

def save_scores(path, scores, pack):
    arrs = {f"doc_{d}": v.astype(np.float16) for d, v in scores.items()}
    arrs["__trans"], arrs["__start"], arrs["__end"] = pack
    np.savez_compressed(path, **arrs)

def ner_loss(model, em, tags, pmask, cw, label_smoothing):
    ce = F.cross_entropy(em[pmask].float(), tags[pmask], weight=cw, label_smoothing=label_smoothing)
    if model.crf is None:
        return ce
    nll = model.crf.nll(em.float(), tags, pmask)
    return nll.sum() / pmask.sum().clamp(min=1) + CRF_AUX_CE_WEIGHT * ce

def load_trained(name):
    run_dir = OUTPUT_DIR / "runs" / name
    cfg = json.loads((run_dir / "cfg.json").read_text())
    model = build_ner_model(cfg)
    state = torch.load(run_dir / "best_state.pt", map_location="cpu")
    res = model.load_state_dict(state, strict=False)
    expected = {n for n, p in model.named_parameters() if p.requires_grad}
    assert expected <= set(state), "saved state lacks some trainable tensors"
    return model, cfg

def kd_losses(model, s_em, t_ems, s_hid, t_hid, tags, pmask, tmask, layer_map, cfg, cw, label_smoothing):
    s = s_em[pmask].float()
    T = cfg["kd_temperature"]
    target = torch.stack([F.softmax(t[pmask].float() / T, -1) for t in t_ems]).mean(0)
    kd = F.kl_div(F.log_softmax(s / T, -1), target, reduction="batchmean") * T * T
    hard = ner_loss(model, s_em, tags, pmask, cw, label_smoothing)
    hid = s_em.new_zeros(()).float()
    if cfg["kd_hidden_weight"] > 0:
        d = s_hid[0].size(-1)
        for si, ti in enumerate(layer_map):
            a = F.layer_norm(s_hid[si].float(), (d,))
            b = F.layer_norm(t_hid[ti].float(), (d,))
            hid = hid + ((a - b) ** 2).mean(-1)[tmask].mean()
        hid = hid / len(layer_map)
    return (1 - cfg["kd_alpha"]) * kd + cfg["kd_alpha"] * hard + cfg["kd_hidden_weight"] * hid, kd, hard, hid

def run_experiment(name, teacher=None):
    cfg = make_cfg(name)
    run_dir = OUTPUT_DIR / "runs" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    if RESUME_COMPLETED_RUNS and (run_dir / "results.json").exists():
        res = json.loads((run_dir / "results.json").read_text())
        RESULTS[name] = res
        print(f"[{name}] already finished (dev macro-F1 {res['dev']['macro_f1']:.4f}) - skipping; "
              f"set RESUME_COMPLETED_RUNS=False to retrain.")
        return res

    print("=" * 100); print(f"{name}: {cfg['description']}"); print("=" * 100)
    seed_everything(SEED)
    window = cfg["window"] or WINDOW
    bs = cfg["batch_size"] or BATCH_SIZE
    accum = cfg["grad_accum"] or GRAD_ACCUM
    is_kd = cfg["method"] == "distill"

    model = build_ner_model(cfg).to(DEVICE)
    n_total, n_train = count_params(model), count_params(model, True)
    print(f"parameters: total={n_total/1e6:.2f}M trainable={n_train/1e6:.3f}M ({100*n_train/n_total:.2f}%) "
          f"| causal={model.causal}")
    assert MIN_PARAMS <= n_total <= MAX_PARAMS or SMOKE_TEST, \
        f"{n_total/1e6:.1f}M parameters outside the {MIN_PARAMS/1e6:.0f}M-{MAX_PARAMS/1e6:.0f}M course limit"

    teachers = [] if teacher is None else (list(teacher) if isinstance(teacher, (list, tuple)) else [teacher])
    if is_kd:
        assert teachers, "distillation needs a teacher"
        for t in teachers:
            t.to(DEVICE).eval()
            for p in t.parameters():
                p.requires_grad = False
        t_params = sum(count_params(t) for t in teachers)
        print(f"teacher: {len(teachers)} model(s), {t_params/1e6:.1f}M parameters in total -> student {n_total/1e6:.1f}M")

    head, body_decay, body_nodecay = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith(("classifier", "lstm", "feat", "crf")):
            head.append(p)
        elif p.ndim >= 2 and cfg["method"] in ("full", "partial", "distill"):
            body_decay.append(p)
        else:
            body_nodecay.append(p)
    groups = [dict(params=head, lr=cfg["head_lr"], weight_decay=0.0)]
    if body_decay:   groups.append(dict(params=body_decay, lr=cfg["lr"], weight_decay=WEIGHT_DECAY))
    if body_nodecay: groups.append(dict(params=body_nodecay, lr=cfg["lr"], weight_decay=0.0))
    optim = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)

    rng = random.Random(SEED)
    first = TrainWindows(train_ids, window, TRAIN_NEG_KEEP, rng)
    steps_per_epoch = max(1, math.ceil(len(first) / (bs * accum)))
    total_steps = steps_per_epoch * cfg["epochs"]
    warm = max(1, int(WARMUP_FRACTION * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        optim, lambda s: min((s + 1) / warm, max(0.0, (total_steps - s) / max(1, total_steps - warm))))
    use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
    scaler = make_grad_scaler(use_amp)
    cw = class_weight_tensor(cfg)
    print(f"train windows/epoch ~{len(first):,} | optimizer steps/epoch ~{steps_per_epoch} | total ~{total_steps}")

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    history, best_f1, best_epoch, bad = [], -1.0, 0, 0
    t_train = time.time()
    ds = first
    for epoch in range(1, cfg["epochs"] + 1):
        if epoch > 1:
            ds = TrainWindows(train_ids, window, TRAIN_NEG_KEEP, rng)
        loader = DataLoader(ds, batch_size=bs, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)
        model.train()
        run_loss, n_loss, micro = 0.0, 0, 0
        optim.zero_grad(set_to_none=True)
        pbar = tqdm(loader, desc=f"{name} ep{epoch}/{cfg['epochs']}", leave=False)
        for batch in pbar:
            batch = to_device(batch)
            args = (batch["ids"], batch["mask"], batch["pos"], batch["pmask"], batch["feats"])
            mem_kw = dict(mem_ids=batch["mem_ids"], mem_mask=batch["mem_mask"])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                if is_kd:
                    em, s_hid = model(*args, return_hidden=True, **mem_kw)
                    with torch.no_grad():
                        t_out = [t(*args, return_hidden=True, **mem_kw) for t in teachers]
                    t_hid = t_out[0][1]
                    loss, kd, hard, hid = kd_losses(model, em, [o[0] for o in t_out], s_hid, t_hid, batch["tags"], batch["pmask"],
                                                    batch["mask"], cfg["layer_map"], cfg, cw, cfg["label_smoothing"])
                else:
                    em = model(*args, **mem_kw)
                    loss = ner_loss(model, em, batch["tags"], batch["pmask"], cw, cfg["label_smoothing"])
            if not torch.isfinite(loss):
                optim.zero_grad(set_to_none=True); continue
            (loss / accum).backward() if scaler is None else scaler.scale(loss / accum).backward()
            micro += 1
            run_loss += float(loss); n_loss += 1
            if micro % accum == 0:
                if scaler is not None:
                    scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], GRAD_CLIP)
                if scaler is None: optim.step()
                else: scaler.step(optim); scaler.update()
                sched.step(); optim.zero_grad(set_to_none=True)
            pbar.set_postfix(loss=f"{run_loss / max(1, n_loss):.4f}")
        if micro % accum != 0:
            if scaler is not None: scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], GRAD_CLIP)
            if scaler is None: optim.step()
            else: scaler.step(optim); scaler.update()
            sched.step(); optim.zero_grad(set_to_none=True)

        scores, _, _, _ = evaluate_docs(model, dev_ids, methods=("viterbi",))
        f1 = scores["viterbi"]["macro_f1"]
        history.append(dict(epoch=epoch, train_loss=run_loss / max(1, n_loss), dev_macro_f1=f1,
                            dev_micro_f1=scores["viterbi"]["micro_f1"], elapsed_s=time.time() - t_train))
        print(f"  epoch {epoch:2d} | loss {history[-1]['train_loss']:.4f} | dev strict macro-F1 {f1:.4f} "
              f"| micro-F1 {scores['viterbi']['micro_f1']:.4f} | {history[-1]['elapsed_s']/60:.1f} min")
        if f1 > best_f1 + 1e-5:
            best_f1, best_epoch, bad = f1, epoch, 0
            torch.save(trainable_state(model), run_dir / "best_state.pt")
        elif best_f1 > 0:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"  early stop (no improvement for {cfg['patience']} epochs)")
                break
    train_time = time.time() - t_train
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else 0.0

    model.load_state_dict(torch.load(run_dir / "best_state.pt", map_location=DEVICE), strict=False)
    scores, preds, speed, dev_sc = evaluate_docs(model, dev_ids, methods=("viterbi", "greedy"), keep_scores=True)
    save_scores(run_dir / "dev_scores.npz", dev_sc, model_transitions(model))
    decoder = max(scores, key=lambda m: scores[m]["macro_f1"])
    dev = scores[decoder]
    print(f"[{name}] best epoch {best_epoch} | decoder={decoder} | dev strict macro-P/R/F1 = "
          f"{dev['macro_precision']:.4f}/{dev['macro_recall']:.4f}/{dev['macro_f1']:.4f} | micro-F1 {dev['micro_f1']:.4f}")
    print(f"        greedy={scores['greedy']['macro_f1']:.4f}  viterbi={scores['viterbi']['macro_f1']:.4f} | "
          f"{speed['tokens_per_s']:,.0f} tok/s")
    save_json(run_dir / "dev_predictions.json", {d: [list(s) for s in v] for d, v in preds[decoder].items()})

    res = dict(name=name, method=cfg["method"], description=cfg["description"], cfg=cfg,
               params_total=n_total, params_trainable=n_train, trainable_pct=100 * n_train / n_total,
               train_time_s=train_time, epochs_run=len(history), best_epoch=best_epoch, peak_mem_gb=peak_gb,
               decoder=decoder, dev=dev, dev_greedy=scores["greedy"], dev_viterbi=scores["viterbi"],
               infer_tokens_per_s=speed["tokens_per_s"],
               state_mb=(run_dir / "best_state.pt").stat().st_size / 1024**2, history=history)

    if WRITE_SUBMISSIONS:
        test_spans, test_sc = predict_test(model, decoder)
        save_scores(run_dir / "test_scores.npz", test_sc, model_transitions(model))
        zp, n_ent = write_submission(run_dir, test_spans, sorted(test_texts), test_texts)
        res["submission_zip"] = str(zp); res["test_entities_predicted"] = n_ent
        print(f"        submission: {zp} ({n_ent:,} predicted entities in {len(test_texts)} test docs)")

    save_json(run_dir / "cfg.json", cfg)
    save_json(run_dir / "results.json", res)
    RESULTS[name] = res
    model.cpu()
    TRAINED_MODELS[name] = model
    for t in teachers:
        t.cpu()
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return res

In [ ]:
def roundtrip_check(doc_ids, limit=400):
    tot = ok = text_ok = 0
    for d in doc_ids[:limit]:
        e = ENC["train"][d]
        sub = spans_to_submission({d: tags_to_spans(e["tags"], e["pieces"])}, [d])[d]
        got = {tuple(x) for x in sub}
        raw = {(s, en, l) for s, en, l, _ in gold_raw[d]}
        tot += len(raw); ok += len(raw & got)
        for s, en, l in got & raw:
            text_ok += 1
    return ok / max(1, tot), tot

rec, tot = roundtrip_check(train_ids)
print(f"Round-trip exact recall of gold spans through the full alignment/decoding path: {rec:.4f} over {tot:,} spans")
print(f"(= the strict-F1 ceiling of this tagging scheme; expected to match the oracle recall printed earlier: {audit[PRETOK]:.4f}, "
      f"which was measured on the first 150 train docs only)")
assert rec > 0.5, "Alignment looks broken - inspect offsets / END_INCLUSIVE before training"
if with_text:
    d0 = next(d for d in train_ids if gold_raw[d])
    s, en, l, tx = gold_raw[d0][0]
    print(f"Example: {l:8s} [{s}, {en}] -> {texts[d0][s:en + (1 if END_INCLUSIVE else 0)]!r}  (gold text: {tx!r})")

In [ ]:
TAPT_PATH = OUTPUT_DIR / f"tapt_encdec_{PRETOK}_{TOKENIZER_SHA256[:8]}_{TAPT_EPOCHS}ep_{int(TAPT_INCLUDE_TEST_TEXT)}.pt"

def _lm_stream(doc_ids, split):
    parts = []
    for d in doc_ids:
        parts.append(ENC[split][d]["ids"]); parts.append(np.array([EOS_ID], dtype=np.int32))
    stream = np.concatenate(parts).astype(np.int64)
    return stream, len(stream) // TAPT_BLOCK

def _pair_batch(stream, idx):
    blocks = np.stack([stream[j * TAPT_BLOCK:(j + 1) * TAPT_BLOCK] for j in idx])
    src, tgt = blocks[:, :TAPT_SOURCE], blocks[:, TAPT_SOURCE:]
    b = len(idx)
    enc = np.concatenate([np.full((b, 1), BOS_ID), src, np.full((b, 1), EOS_ID)], 1)
    dec = np.concatenate([np.full((b, 1), BOS_ID), tgt[:, :-1]], 1)
    return (torch.from_numpy(enc).to(DEVICE), torch.from_numpy(dec).to(DEVICE), torch.from_numpy(tgt).to(DEVICE))

def _seq2seq_loss(bb, enc, dec, tgt, grad_ckpt=False):
    ones_e = torch.ones_like(enc, dtype=torch.bool)
    ones_d = torch.ones_like(dec, dtype=torch.bool)
    mem, _ = bb.encode(enc, ones_e, grad_ckpt=grad_ckpt)
    h, _ = bb.decode(dec, mem, ones_e, ones_d, grad_ckpt=grad_ckpt)
    logits = F.linear(h, bb.shared_embedding.weight)
    return F.cross_entropy(logits.float().view(-1, logits.size(-1)), tgt.reshape(-1))

@torch.no_grad()
def seq2seq_perplexity(bb, stream, n_blocks, max_blocks=120):
    bb.eval()
    tot = cnt = 0.0
    for i in range(0, min(n_blocks, max_blocks), TAPT_BATCH):
        idx = list(range(i, min(i + TAPT_BATCH, n_blocks, max_blocks)))
        enc, dec, tgt = _pair_batch(stream, idx)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            loss = _seq2seq_loss(bb, enc, dec, tgt)
        tot += loss.item() * tgt.numel(); cnt += tgt.numel()
    return math.exp(tot / max(1, cnt))

def run_tapt():
    global P1_STATE
    if not RUN_TAPT:
        print("TAPT disabled - using the Phase-1 weights as they are.")
        return None
    if TAPT_PATH.exists():
        P1_STATE = torch.load(TAPT_PATH, map_location="cpu")
        print("Loaded task-adapted backbone:", TAPT_PATH.name)
        info_path = TAPT_PATH.with_suffix(".json")
        return json.loads(info_path.read_text()) if info_path.exists() else None
    seed_everything(SEED)
    bb = build_backbone().to(DEVICE)
    tr_stream, tr_n = _lm_stream(train_ids, "train")
    if TAPT_INCLUDE_TEST_TEXT:
        te_stream, _ = _lm_stream(sorted(ENC["test"]), "test")
        tr_stream = np.concatenate([tr_stream, te_stream]); tr_n = len(tr_stream) // TAPT_BLOCK
    dv_stream, dv_n = _lm_stream(dev_ids, "train")
    ppl_before = seq2seq_perplexity(bb, dv_stream, dv_n)
    per_epoch = tr_n if not TAPT_MAX_TOKENS_PER_EPOCH else min(tr_n, TAPT_MAX_TOKENS_PER_EPOCH // TAPT_BLOCK)
    total = max(1, TAPT_EPOCHS * math.ceil(per_epoch / TAPT_BATCH))
    warm = max(1, int(0.05 * total))
    opt = torch.optim.AdamW(bb.parameters(), lr=TAPT_LR, weight_decay=0.01, betas=(0.9, 0.98))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.05, (total - s) / max(1, total - warm))))
    use_amp = DEVICE.type == "cuda"
    scaler = make_grad_scaler(use_amp)
    print(f"TAPT: {tr_n:,} blocks ({TAPT_SOURCE} source -> {TAPT_BLOCK - TAPT_SOURCE} target tokens) | {TAPT_EPOCHS} epochs | "
          f"dev perplexity before = {ppl_before:.2f}")
    rng = np.random.default_rng(SEED)
    t0 = time.time()
    for ep in range(TAPT_EPOCHS):
        bb.train()
        order = rng.permutation(tr_n)[:per_epoch]
        run = 0.0
        pbar = tqdm(range(0, len(order), TAPT_BATCH), desc=f"TAPT ep{ep + 1}", leave=False)
        for i in pbar:
            enc, dec, tgt = _pair_batch(tr_stream, order[i:i + TAPT_BATCH])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                loss = _seq2seq_loss(bb, enc, dec, tgt, grad_ckpt=USE_GRAD_CHECKPOINTING)
            opt.zero_grad(set_to_none=True)
            if scaler is None:
                loss.backward(); torch.nn.utils.clip_grad_norm_(bb.parameters(), 1.0); opt.step()
            else:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(bb.parameters(), 1.0); scaler.step(opt); scaler.update()
            sch.step()
            run = 0.98 * run + 0.02 * loss.item() if run else loss.item()
            pbar.set_postfix(loss=f"{run:.3f}")
        print(f"  TAPT epoch {ep + 1}: dev perplexity {seq2seq_perplexity(bb, dv_stream, dv_n):.2f} | "
              f"{(time.time() - t0) / 60:.1f} min")
    ppl_after = seq2seq_perplexity(bb, dv_stream, dv_n)
    new_state = {k: v.detach().cpu() for k, v in bb.state_dict().items()}
    for k, v in P1_STATE.items():
        if k not in new_state:
            new_state[k] = v
    P1_STATE = new_state
    torch.save(P1_STATE, TAPT_PATH)
    info = dict(ppl_before=ppl_before, ppl_after=ppl_after, epochs=TAPT_EPOCHS, minutes=(time.time() - t0) / 60,
                train_blocks=tr_n, include_test_text=TAPT_INCLUDE_TEST_TEXT)
    save_json(TAPT_PATH.with_suffix(".json"), info)
    del bb, opt
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print(f"TAPT done: dev perplexity {ppl_before:.2f} -> {ppl_after:.2f}. All experiments below start from these weights.")
    return info

TAPT_INFO = run_tapt()

In [ ]:
TEACHER_CANDIDATES = ["E1_full_ft", "E2_lora", "E3_lorma_plus"]

for _p in sorted((OUTPUT_DIR / "runs").glob("*/results.json")):
    RESULTS[_p.parent.name] = json.loads(_p.read_text())

def pick_teachers(k):
    done = {n: RESULTS[n]["dev"]["macro_f1"] for n in TEACHER_CANDIDATES if n in RESULTS}
    if not done:
        raise RuntimeError("Distillation needs finished teachers: run E1/E2/E3 first.")
    names = sorted(done, key=done.get, reverse=True)[:k]
    if len(names) < k:
        print(f"Note: asked for {k} teachers, {len(names)} finished run(s) available.")
    return names, [TRAINED_MODELS.get(n) or load_trained(n)[0] for n in names]

for name in RUN_LIST:
    teacher = None
    if EXPERIMENTS[name]["method"] == "distill":
        tnames, teacher = pick_teachers(EXPERIMENTS[name].get("teacher_ensemble", 1))
        EXPERIMENTS[name]["teacher_name"] = "+".join(tnames)
        print(f"Teacher ensemble for {name}: " + ", ".join(
            f"{n} (dev macro-F1 {RESULTS[n]['dev']['macro_f1']:.4f})" for n in tnames))
    run_experiment(name, teacher)
    gc.collect()

print("\nFinished:", [n for n in RUN_LIST if n in RESULTS])

In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

ORDER = [n for n in EXPERIMENTS if n in RESULTS]
rows = []
for n in ORDER:
    r = RESULTS[n]
    rows.append(dict(
        experiment=n, method=r["method"], head=r["cfg"].get("head", "linear"), params_M=r["params_total"] / 1e6, trainable_M=r["params_trainable"] / 1e6,
        trainable_pct=r["trainable_pct"], dev_macro_F1=r["dev"]["macro_f1"], dev_macro_P=r["dev"]["macro_precision"],
        dev_macro_R=r["dev"]["macro_recall"], dev_micro_F1=r["dev"]["micro_f1"], decoder=r["decoder"],
        greedy_F1=r["dev_greedy"]["macro_f1"], viterbi_F1=r["dev_viterbi"]["macro_f1"], best_epoch=r["best_epoch"],
        train_min=r["train_time_s"] / 60, peak_mem_GB=r["peak_mem_gb"], infer_tok_per_s=r["infer_tokens_per_s"],
        ckpt_MB=r["state_mb"]))
summary = pd.DataFrame(rows).set_index("experiment")
summary.to_csv(OUTPUT_DIR / "phase2_results_summary.csv")
display(summary.round(4))

per_label = pd.DataFrame({n: {l: RESULTS[n]["dev"]["per_label"][l]["f1"] for l in LABELS} for n in ORDER})
per_label["support"] = [RESULTS[ORDER[0]]["dev"]["per_label"][l]["support"] for l in LABELS]
display(per_label.round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
summary["dev_macro_F1"].plot.bar(ax=ax[0], color="#4C72B0"); ax[0].set_title("Dev strict macro-F1"); ax[0].set_ylabel("F1")
ax[0].tick_params(axis="x", rotation=30)
for n in ORDER:
    h = RESULTS[n]["history"]
    ax[1].plot([x["epoch"] for x in h], [x["dev_macro_f1"] for x in h], marker="o", label=n)
ax[1].set_title("Dev macro-F1 per epoch"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "phase2_overview.png", dpi=130); plt.show()

def error_breakdown(pred, gold):
    cats, confusion, d_start, d_end = Counter(), Counter(), [], []
    for d in gold:
        G = list(gold[d]); P = [tuple(x) for x in pred.get(d, [])]
        gset, pset = set(G), set(P)
        cats["exact match (TP)"] += len(gset & pset)
        for p in pset - gset:
            ov = [g for g in G if g[0] < p[1] and p[0] < g[1]]
            if not ov:
                cats["spurious prediction (no overlap)"] += 1; continue
            same_span = [g for g in ov if g[0] == p[0] and g[1] == p[1]]
            if same_span:
                cats["label confusion (exact span)"] += 1; confusion[(same_span[0][2], p[2])] += 1; continue
            same_lab = [g for g in ov if g[2] == p[2]]
            if same_lab:
                g = max(same_lab, key=lambda g: min(g[1], p[1]) - max(g[0], p[0]))
                cats["boundary error (right label)"] += 1; d_start.append(p[0] - g[0]); d_end.append(p[1] - g[1])
            else:
                cats["overlap, wrong label AND boundary"] += 1
        for g in gset - pset:
            if not any(p[0] < g[1] and g[0] < p[1] for p in pset):
                cats["missed entity (no overlap)"] += 1
    return cats, confusion, d_start, d_end

best_name = max(ORDER, key=lambda n: RESULTS[n]["dev"]["macro_f1"])
dev_pred = {d: [tuple(x) for x in v] for d, v in json.loads((OUTPUT_DIR / "runs" / best_name / "dev_predictions.json").read_text()).items()}
dev_gold = {d: gold_all[d] for d in dev_ids}
cats, confusion, d_start, d_end = error_breakdown(dev_pred, dev_gold)
print(f"Error breakdown on dev for the best run: {best_name}")
display(pd.Series(cats, name="count").to_frame())
if d_start:
    ds_, de_ = np.array(d_start), np.array(d_end)
    print(f"Boundary errors: start off by {np.mean(ds_ != 0):.0%} / end off by {np.mean(de_ != 0):.0%} of them; "
          f"median |start shift|={np.median(np.abs(ds_)):.0f} chars, median |end shift|={np.median(np.abs(de_)):.0f} chars")
    print("Most common (start shift, end shift):", Counter(zip(ds_.tolist(), de_.tolist())).most_common(6))
if confusion:
    print("Most confused label pairs (gold -> predicted):")
    for (g, p), c in confusion.most_common(8):
        print(f"   {g:10s} -> {p:10s} {c}")

def lenient_f1(pred, gold):
    tp = fp = fn = 0
    for d in gold:
        G, P = gold[d], [tuple(x) for x in pred.get(d, [])]
        for p in P:
            tp_ = any(g[2] == p[2] and g[0] < p[1] and p[0] < g[1] for g in G)
            tp += tp_; fp += (not tp_)
        fn += sum(not any(g[2] == p[2] and g[0] < p[1] and p[0] < g[1] for p in P) for g in G)
    pr = tp / max(1, tp + fp); rc = tp / max(1, tp + fn)
    return 2 * pr * rc / max(1e-9, pr + rc)

print(f"\nBest run micro-F1: strict={RESULTS[best_name]['dev']['micro_f1']:.4f} | overlap-lenient={lenient_f1(dev_pred, dev_gold):.4f} "
      f"(the gap is what boundary errors cost you)")

lens_dev = {d: len(ENC["train"][d]["ids"]) for d in dev_ids}
order = sorted(dev_ids, key=lens_dev.get)
terciles = np.array_split(np.array(order, dtype=object), 3)
print("\nMicro-F1 by dev-document length tercile (best run):")
for name_, grp in zip(("short", "medium", "long"), terciles):
    if len(grp) == 0:
        continue
    sc = strict_scores({d: dev_pred.get(d, []) for d in grp}, {d: dev_gold[d] for d in grp})
    print(f"  {name_:7s} docs={len(grp):4d} tokens<= {max(lens_dev[d] for d in grp):7,d}  micro-F1={sc['micro_f1']:.4f}")

print("\n--- Key comparisons for the written analysis ---")
if "E1_full_ft" in RESULTS and RESULTS["E1_full_ft"]["dev"]["macro_f1"] > 0:
    full = RESULTS["E1_full_ft"]["dev"]["macro_f1"]
    for n in ("E2_lora", "E3_lorma_plus"):
        if n in RESULTS:
            r = RESULTS[n]
            print(f"{n:22s}: {r['dev']['macro_f1']:.4f} = {r['dev']['macro_f1']/full:.1%} of full-FT F1 with "
                  f"{r['trainable_pct']:.2f}% of parameters trainable")
    if "E4_distill" in RESULTS:
        r = RESULTS["E4_distill"]
        tn = [n for n in r["cfg"].get("teacher_name", "").split("+") if n in RESULTS]
        if tn and RESULTS[tn[0]]["dev"]["macro_f1"] > 0:
            tp = sum(RESULTS[n]["params_total"] for n in tn)
            best_t = max(RESULTS[n]["dev"]["macro_f1"] for n in tn)
            t0 = RESULTS[tn[0]]
            print(f"E4 student ({r['params_total']/1e6:.1f}M) vs teacher {'+'.join(tn)} ({tp/1e6:.1f}M): "
                  f"student F1 {r['dev']['macro_f1']:.4f} = {r['dev']['macro_f1']/best_t:.1%} of the teacher's; "
                  f"{100 * (1 - r['params_total'] / tp):.0f}% fewer parameters; "
                  f"speed {r['infer_tokens_per_s'] / max(1e-9, t0['infer_tokens_per_s']):.2f}x")
if TAPT_INFO:
    print(f"TAPT: dev LM perplexity {TAPT_INFO['ppl_before']:.2f} -> {TAPT_INFO['ppl_after']:.2f}")
print(f"Oracle (pre-tokeniser '{PRETOK}') exact-boundary ceiling: {audit[PRETOK]:.4f} | other mode: {audit['whitespace' if PRETOK=='punct' else 'punct']:.4f}")

In [ ]:
def load_scores(name, split):
    z = np.load(OUTPUT_DIR / "runs" / name / f"{split}_scores.npz")
    sc = {k[4:]: z[k].astype(np.float32) for k in z.files if k.startswith("doc_")}
    return sc, (z["__trans"], z["__start"], z["__end"])

def _logsoftmax(a):
    m = a.max(-1, keepdims=True)
    return a - m - np.log(np.exp(a - m).sum(-1, keepdims=True))

def combine(members, split):
    loaded = [load_scores(n, split) for n in members]
    docs = loaded[0][0].keys()
    sc = {d: np.mean([_logsoftmax(L[0][d]) for L in loaded], axis=0) for d in docs}
    pack = tuple(np.mean([L[1][i] for L in loaded], axis=0) for i in range(3))
    return sc, pack

def ensemble_dev_f1(members):
    sc, pack = combine(members, "dev")
    pred = {d: decode_spans(sc[d], ENC["train"][d]["pieces"], "viterbi", pack) for d in dev_ids}
    return strict_scores(pred, {d: gold_all[d] for d in dev_ids}), pred

In [ ]:
SUB_DIR = OUTPUT_DIR / "submissions"
SUB_DIR.mkdir(exist_ok=True)
import shutil
print(f"{'experiment':28s} {'dev macro-F1':>12s}  zip")
for n in ORDER:
    zp = OUTPUT_DIR / "runs" / n / "submission.zip"
    if not zp.exists():
        continue
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
        with z.open("prediction.json") as f:
            sub = json.load(f)
    assert set(sub) == set(test_texts)
    dest = SUB_DIR / f"phase2_{n}.zip"
    shutil.copy(zp, dest)
    print(f"{n:28s} {RESULTS[n]['dev']['macro_f1']:12.4f}  {dest}  "
          f"({sum(len(v) for v in sub.values()):,} entities)")

print("\nUpload each phase2_*.zip on Codabench (Phase 2). The course needs >= 4 valid submissions per phase;")
print("record the Codabench test score next to each dev score in your report. Dev scores are from a held-out slice of")
print("the labelled folds, so expect the leaderboard number to differ.")
print(f"End-offset convention used in the files: {'inclusive' if END_INCLUSIVE else 'exclusive'} (mirrors the gold files).")